# 03 — Joint nanoGPT-style Route Generation

## From Understanding to Generation

Notebook 02 used a **transformer encoder** (BERT-style) to *understand* routes and predict their grade. This notebook uses a **transformer decoder** (GPT-style) to *generate* new routes.

### The key difference: Encoder vs Decoder

| Aspect | BERT-style (Encoder) | GPT-style (Decoder) |
|---|---|---|
| Attention | Bidirectional (sees all tokens) | Causal (only sees past tokens) |
| Training | Masked language modeling | Next-token prediction |
| Use case | Classification, regression | Text generation |
| Output | Single prediction per sequence | One prediction per position |

### How GPT-style generation works

The model is trained to predict the **next token** given all previous tokens:

```text
Input:  <BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6>
Target: <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start>
```

At generation time, we:
1. Start with a prompt like `<BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6>`
2. Ask the model to predict the next token
3. Sample from the predicted probability distribution
4. Append the sampled token to the sequence
5. Repeat until we generate `<EOS>` or hit a max length

### Conditioning on board, angle, and grade

The prompt tokens tell the model *what kind of route to generate*:
- `<BOARD_TB2>`: Generate a route for the Tension Board 2
- `<ANGLE_40>`: At 40 degrees
- `<GRADE_V6>`: At V6 difficulty

This is analogous to how ChatGPT uses a system prompt to condition its responses.



In [ ]:
from __future__ import annotations

import ast
import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
TOKENIZED = ROOT / "data" / "processed" / "tokenized"
df_routes = pd.read_csv(TOKENIZED / "route_sequences.csv")
vocab = json.loads((TOKENIZED / "token_vocab.json").read_text(encoding="utf-8"))
stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
itos = {int(k): str(v) for k, v in vocab["itos"].items()}

pad_id = stoi["<PAD>"]
unk_id = stoi["<UNK>"]

print(f"Vocabulary size: {len(stoi):,}")
print(f"Total routes: {len(df_routes):,}")

### Causal dataset helper

In [ ]:
# Pad route-token sequences and create shifted input/target pairs for causal modeling.
class RouteGPTDataset(Dataset):
    """Dataset for causal next-token route generation.

    The full sequence is padded once, then split into ``input_ids`` and
    ``target_ids`` shifted by one position for teacher-forced language-model
    training.
    """

    def __init__(self, df, max_len: int, pad_id: int):
        """Store GPT token ID sequences from a tokenized route DataFrame."""
        self.ids = df["gpt_ids"].tolist()
        self.max_len = int(max_len)
        self.pad_id = int(pad_id)

    def __len__(self) -> int:
        """Return the number of route examples."""
        return len(self.ids)

    def __getitem__(self, idx: int):
        """Return one padded causal-language-model training example."""
        ids = list(self.ids[idx])[: self.max_len]
        if len(ids) < self.max_len:
            ids += [self.pad_id] * (self.max_len - len(ids))

        return {
            "input_ids": torch.tensor(ids[:-1], dtype=torch.long),
            "target_ids": torch.tensor(ids[1:], dtype=torch.long),
        }

## Sequence encoding for causal language modeling

### The autoregressive setup

For GPT-style training, each route becomes a sequence where the model learns to predict each token given all previous tokens:

```text
Input:   <BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start> <TB2_p369_middle>
Target:  <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start> <TB2_p369_middle> <TB2_p603_finish>
```

The input is shifted right by one position compared to the target. This is the standard causal language modeling setup.

### Why include the grade in the training sequence?

For the grade predictor (notebook 02), we excluded the grade because the model needed to predict it. But for the generator, we **include** the grade (`<GRADE_V6>`) in the training data so the model learns the relationship between grade and hold selection.

At generation time, we provide the grade as part of the prompt, and the model generates holds that are appropriate for that grade.



In [ ]:
def encode(tokens):
    """Convert token strings to integer IDs."""
    return [stoi.get(token, unk_id) for token in tokens]

# Use the "with grade" version for GPT training
# The model needs to see the grade to learn grade-hold relationships
df_routes["gpt_tokens"] = df_routes["sequence_with_grade"].fillna("").str.split()
df_routes["gpt_ids"] = df_routes["gpt_tokens"].apply(encode)
df_routes["seq_len"] = df_routes["gpt_ids"].apply(len)
max_len = int(df_routes["seq_len"].max())
block_size = max_len - 1  # Input length (one less than full sequence)

# Create train/val splits
train_df = df_routes[df_routes["split"] == "train"].reset_index(drop=True)
val_df = df_routes[df_routes["split"] == "val"].reset_index(drop=True)

# Create datasets and data loaders
# RouteGPTDataset handles the input/target shift for causal modeling
train_ds = RouteGPTDataset(train_df, max_len=max_len, pad_id=pad_id)
val_ds = RouteGPTDataset(val_df, max_len=max_len, pad_id=pad_id)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

print(f"Max sequence length: {max_len}")
print(f"Block size (input length): {block_size}")
print(f"Training samples: {len(train_ds):,}")
print(f"Validation samples: {len(val_ds):,}")

### GPT model

In [ ]:
# GPT-style causal transformer for route generation.
class JointRouteGPT(nn.Module):
    """Tiny GPT-style causal transformer for board-conditioned route generation.

    PyTorch's ``TransformerEncoder`` is used with a causal mask, which makes it
    behave like a decoder-only language model for short route sequences.

    Why use ``TransformerEncoder`` rather than ``TransformerDecoder``?
    -------------------------------------------------------------------
    PyTorch's ``TransformerDecoderLayer`` expects two inputs: a decoder
    sequence and a separate encoder memory for cross-attention. For
    unconditional or prompt-conditioned generation there is no encoder,
    so ``TransformerDecoderLayer`` would always ignore the second input
    or require a dummy placeholder. Using ``TransformerEncoder`` with a
    causal mask avoids this mismatch, keeps the module list uniform,
    and produces identical behaviour for short autoregressive generation.

    The trade-off is that ``TransformerEncoder`` does not natively prevent
    attention to future positions — the causal mask must be constructed
    manually (see ``forward``). For the sequence lengths seen here
    (at most ~400 tokens) the overhead of the upper-triangular mask is
    negligible, and ``enable_nested_tensor=False`` is set to avoid SDPA
    optimisations that do not support masked encoders.
    """

    def __init__(
        self,
        vocab_size: int,
        block_size: int,
        n_embd: int = 128,
        n_head: int = 4,
        n_layer: int = 4,
        dropout: float = 0.10,
        pad_id: int = 0,
    ):
        """Create the token/position embeddings, causal blocks, and LM head."""
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.pad_id = pad_id

        self.token_emb = nn.Embedding(vocab_size, n_embd, padding_idx=pad_id)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            dim_feedforward=4 * n_embd,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(
            layer,
            num_layers=n_layer,
            enable_nested_tensor=False,
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward(
        self,
        idx: torch.Tensor,
        targets: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Return next-token logits and, when targets are supplied, CE loss."""
        _, seq_len = idx.shape
        if seq_len > self.block_size:
            idx = idx[:, -self.block_size :]
            seq_len = idx.shape[1]

        positions = torch.arange(seq_len, device=idx.device).unsqueeze(0)
        x = self.drop(self.token_emb(idx) + self.pos_emb(positions))

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=idx.device, dtype=torch.bool),
            diagonal=1,
        )
        # Padding masks suppress attention to right-padded context tokens while
        # the causal mask suppresses attention to future positions.
        key_padding_mask = idx.eq(self.pad_id)

        h = self.blocks(
            x,
            mask=causal_mask,
            src_key_padding_mask=key_padding_mask,
        )
        h = self.ln_f(h)
        logits = self.lm_head(h)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=self.pad_id,
            )

        return logits, loss

## The GPT Model Architecture

### JointRouteGPT

This is a **causal transformer decoder** — the same architecture used in GPT-2, GPT-3, etc., but much smaller:

1. **Token embeddings**: Convert integer token IDs to dense vectors
2. **Positional embeddings**: Learned position vectors (not sinusoidal)
3. **Causal self-attention**: Each position can only attend to previous positions (via a causal mask)
4. **Transformer layers**: Multiple layers of attention + feedforward
5. **Language modeling head**: Projects hidden states to vocabulary logits

### Key hyperparameters

- `n_embd=128`: Embedding dimension (GPT-2 small uses 768)
- `n_head=4`: Number of attention heads
- `n_layer=4`: Number of transformer layers (GPT-2 small uses 12)
- `dropout=0.10`: Dropout probability

This is intentionally small — we're training on a few hundred thousand short sequences, not billions of long documents.

### Weight tying

The output projection layer shares weights with the token embedding layer (`self.lm_head.weight = self.token_emb.weight`). This is a common technique that:
- Reduces parameter count
- Acts as a regularizer
- Is used in GPT-2 and many other language models



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = JointRouteGPT(
    vocab_size=len(stoi),
    block_size=block_size,
    n_embd=128,
    n_head=4,
    n_layer=4,
    dropout=0.10,
    pad_id=pad_id,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

print(f"Device: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def train_epoch():
    """Train for one epoch."""
    model.train()
    losses = []
    n = 0
    for batch in train_loader:
        x = batch["input_ids"].to(device)
        y = batch["target_ids"].to(device)
        
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        
        # Gradient clipping prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        losses.append(loss.item() * x.size(0))
        n += x.size(0)
    return sum(losses) / max(1, n)

@torch.no_grad()
def eval_loss(loader):
    """Evaluate loss on a data loader."""
    model.eval()
    losses = []
    n = 0
    for batch in loader:
        x = batch["input_ids"].to(device)
        y = batch["target_ids"].to(device)
        _, loss = model(x, y)
        losses.append(loss.item() * x.size(0))
        n += x.size(0)
    return sum(losses) / max(1, n)

## Training

### What we're optimizing

The model minimizes **cross-entropy loss** — the standard loss function for language modeling. At each position, the model outputs a probability distribution over the entire vocabulary, and the loss measures how surprised it is by the actual next token.

### Perplexity

We also track **perplexity**, which is `exp(loss)`. Perplexity answers the question: "On average, how many tokens was the model choosing between at each step?" Lower perplexity = better model.

For reference:
- A model that always predicts the right token has perplexity = 1
- A model that picks uniformly from a 1000-token vocab has perplexity = 1000
- Good language models on English text achieve perplexity ~15-20

Our vocabulary is ~4000+ tokens, so a perplexity significantly below that indicates the model is learning meaningful patterns.



In [ ]:
history = []
best_val_loss = float("inf")
best_state = None
patience = 10
stagnant = 0

print("Starting GPT training...\n")

for epoch in range(1, 21):
    train_loss = train_epoch()
    val_loss = eval_loss(val_loader)
    
    # Track perplexity (exponentiated loss)
    train_ppl = math.exp(min(train_loss, 20))
    val_ppl = math.exp(min(val_loss, 20))
    
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_perplexity": train_ppl,
        "val_perplexity": val_ppl,
    })
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stagnant = 0
    else:
        stagnant += 1
    
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val PPL: {val_ppl:.1f}")
    
    if stagnant >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Load best model
if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nBest validation loss: {best_val_loss:.4f}")
print(f"Best validation perplexity: {math.exp(min(best_val_loss, 20)):.1f}")

### Board configuration helpers

In [ ]:
# Find the project root and load board configuration JSON files.
def find_project_root(start: str | Path | None = None) -> Path:
    """Walk upward until the repository root markers are found.

    The project root is identified by both ``pyproject.toml`` and ``configs``.
    If neither marker pair is found, the resolved starting directory is returned
    so callers still have a deterministic base path.
    """
    current = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").exists():
            return candidate
    return current

@dataclass(frozen=True)
class BoardConfig:
    """Configuration for a single climbing board.
    
    This dataclass stores all board-specific settings needed for
    data loading, tokenization, and model training.
    
    Attributes:
        board_key: Short identifier (e.g., "tb2", "kilter")
        display_name: Human-readable name (e.g., "Tension Board 2 Mirror")
        token_prefix: Namespace for hold tokens (e.g., "TB2", "KILTER")
        db_path: Path to the SQLite database
        layout_id: Which layout in the database to use
        max_angle: Filter out routes steeper than this (None = no filter)
        min_fa_date: Filter out routes first ascended before this date
        placement_y_max: Filter out placements above this Y coordinate
        include_mirror_placement_id: Whether to include mirror info (TB2 only)
        role_definitions: Maps semantic role names to numeric IDs
        boardlib_database_command: Command to download the database
        boardlib_images_command: Command to download board images
        notes: Additional notes about the configuration
    """
    board_key: str
    display_name: str
    token_prefix: str
    db_path: Path
    layout_id: int
    max_angle: float | None
    min_fa_date: str | None
    placement_y_max: float | None
    include_mirror_placement_id: bool
    role_definitions: dict[str, int]
    boardlib_database_command: str | None = None
    boardlib_images_command: str | None = None
    notes: tuple[str, ...] = ()

    @property
    def role_id_to_name(self) -> dict[int, str]:
        """Reverse mapping from numeric role IDs to semantic role names.
        
        Example: {5: 'start', 6: 'middle', 7: 'finish', 8: 'foot'} for TB2
        """
        return {int(role_id): name for name, role_id in self.role_definitions.items()}

    @property
    def board_token(self) -> str:
        """The special token representing this board.
        
        Example: "<BOARD_TB2>" or "<BOARD_KILTER>"
        """
        return f"<BOARD_{self.token_prefix}>"

    def resolve_db_path(self, project_root: Path | None = None) -> Path:
        """Resolve the database path relative to the project root.
        
        If db_path is absolute, return it as-is.
        Otherwise, resolve it relative to the project root.
        """
        project_root = project_root or find_project_root()
        return self.db_path if self.db_path.is_absolute() else project_root / self.db_path

def load_board_config(board_key: str, config_dir: str | Path | None = None) -> BoardConfig:
    """Load a single board configuration from a JSON file.
    
    Args:
        board_key: Board identifier (e.g., "tb2", "kilter")
        config_dir: Directory containing config JSON files
        
    Returns:
        BoardConfig dataclass with all board settings
        
    Raises:
        FileNotFoundError: If the config file doesn't exist
    """
    project_root = find_project_root()
    config_dir = Path(config_dir) if config_dir is not None else project_root / "configs"
    path = config_dir / f"{board_key}.json"
    if not path.exists():
        available = sorted(p.stem for p in config_dir.glob("*.json"))
        raise FileNotFoundError(
            f"Unknown board config '{board_key}'. Available: {available}"
        )

    payload = json.loads(path.read_text(encoding="utf-8"))
    return BoardConfig(
        board_key=str(payload["board_key"]),
        display_name=str(payload["display_name"]),
        token_prefix=str(payload["token_prefix"]),
        db_path=Path(payload["db_path"]),
        layout_id=int(payload["layout_id"]),
        max_angle=None if payload.get("max_angle") is None else float(payload["max_angle"]),
        min_fa_date=payload.get("min_fa_date"),
        placement_y_max=None if payload.get("placement_y_max") is None else float(payload["placement_y_max"]),
        include_mirror_placement_id=bool(payload.get("include_mirror_placement_id", False)),
        role_definitions={str(k): int(v) for k, v in payload["role_definitions"].items()},
        boardlib_database_command=payload.get("boardlib_database_command"),
        boardlib_images_command=payload.get("boardlib_images_command"),
        notes=tuple(payload.get("notes", [])),
    )

def load_board_configs(board_keys: list[str] | tuple[str, ...]) -> list[BoardConfig]:
    """Load multiple board configurations.
    
    Args:
        board_keys: List of board identifiers
        
    Returns:
        List of BoardConfig dataclasses
    """
    return [load_board_config(board_key) for board_key in board_keys]

### Generation helpers

In [ ]:
# Parse generated hold tokens back into structured hold records.
HOLD_TOKEN_PATTERN = re.compile(r"^<([A-Z0-9_]+)_p(\d+)_(start|middle|finish|foot|unknown)>$")

def tokens_to_hold_records(tokens: Iterable[str]) -> list[dict[str, object]]:
    """Extract hold records from model tokens using the shared hold-token grammar."""
    rows: list[dict[str, object]] = []
    for token in tokens:
        match = HOLD_TOKEN_PATTERN.match(str(token))
        if match is None:
            continue
        board_prefix = match.group(1)
        rows.append(
            {
                "token": str(token),
                "board_token_prefix": board_prefix,
                "board_prefix": board_prefix,
                "placement_id": int(match.group(2)),
                "role": match.group(3),
            }
        )
    return rows

# Sample routes from the trained GPT model and convert them back to frames strings.
def top_k_filter(logits: torch.Tensor, k: int | None) -> torch.Tensor:
    """Mask logits outside the top ``k`` choices for each batch row."""
    if k is None or k <= 0 or k >= logits.size(-1):
        return logits
    values, _ = torch.topk(logits, k)
    cutoff = values[:, [-1]]
    return torch.where(logits < cutoff, torch.full_like(logits, -float("inf")), logits)

@torch.no_grad()
def sample_ids(
    model,
    prompt_ids: list[int],
    device: torch.device,
    max_new_tokens: int = 40,
    temperature: float = 0.9,
    top_k: int | None = 50,
    eos_id: int | None = None,
    forbidden_ids: Iterable[int] | None = None,
) -> list[int]:
    """Autoregressively sample token IDs from a trained route generator.

    The returned list includes the prompt IDs and all sampled IDs up to either
    ``max_new_tokens`` or the first sampled ``eos_id``.
    """
    model.eval()
    sequence = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    forbidden_ids = set(forbidden_ids or [])

    for _ in range(max_new_tokens):
        idx_cond = sequence[:, -model.block_size :]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-6)

        # Special tokens like <PAD> and <CLS> are valid vocabulary entries but
        # should never be emitted in the middle of a generated climb.
        for token_id in forbidden_ids:
            logits[:, int(token_id)] = -float("inf")

        logits = top_k_filter(logits, top_k)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        sequence = torch.cat([sequence, next_id], dim=1)

        if eos_id is not None and int(next_id.item()) == int(eos_id):
            break

    return sequence[0].detach().cpu().tolist()

def prompt_tokens(board_prefix: str, angle: int, grouped_v: int) -> list[str]:
    """Build the conditioning prefix used before sampling hold tokens."""
    return [
        "<BOS>",
        f"<BOARD_{board_prefix}>",
        f"<ANGLE_{int(angle)}>",
        f"<GRADE_V{int(grouped_v)}>",
    ]

def hold_records(tokens: Iterable[str]) -> list[dict[str, object]]:
    """Extract hold records from generated tokens."""
    return tokens_to_hold_records(tokens)

def validity_summary(tokens: Iterable[str], requested_board_prefix: str | None = None) -> dict[str, object]:
    """Summarize basic structural validity for generated token sequences."""
    records = hold_records(tokens)
    placements = [record["placement_id"] for record in records]
    roles = [record["role"] for record in records]
    prefixes = [record["board_prefix"] for record in records]

    one_board_only = len(set(prefixes)) <= 1
    matches_requested_board = requested_board_prefix is None or all(prefix == requested_board_prefix for prefix in prefixes)
    no_duplicates = len(placements) == len(set(placements))
    has_start = "start" in roles
    has_finish = "finish" in roles
    enough_holds = len(records) >= 3

    return {
        "n_hold_tokens": len(records),
        "n_unique_placements": len(set(placements)),
        "has_duplicate_placements": not no_duplicates,
        "one_board_only": one_board_only,
        "matches_requested_board": matches_requested_board,
        "has_start": has_start,
        "has_middle": "middle" in roles,
        "has_finish": has_finish,
        "n_start": roles.count("start"),
        "n_middle": roles.count("middle"),
        "n_foot": roles.count("foot"),
        "n_finish": roles.count("finish"),
        "basic_valid": bool(one_board_only and matches_requested_board and no_duplicates and has_start and has_finish and enough_holds),
    }

def generated_tokens_to_frames(tokens: Iterable[str], role_name_to_id: dict[str, int], board_prefix: str | None = None) -> str:
    """Convert generated hold tokens back into a frames string.

    Duplicate placements and unknown roles are skipped, matching the forgiving
    cleanup used by the demo scripts and webapp.
    """
    pieces = []
    seen = set()
    for record in hold_records(tokens):
        if board_prefix is not None and str(record["board_prefix"]) != board_prefix:
            continue
        placement_id = int(record["placement_id"])
        role = str(record["role"])
        if placement_id in seen or role not in role_name_to_id:
            continue
        seen.add(placement_id)
        pieces.append(f"p{placement_id}r{int(role_name_to_id[role])}")
    return "".join(pieces)

def generate_one(
    model,
    stoi: dict[str, int],
    itos: dict[int, str],
    device: torch.device,
    board_prefix: str,
    angle: int,
    grouped_v: int,
    role_name_to_id: dict[str, int],
    temperature: float = 0.9,
    top_k: int | None = 50,
    max_new_tokens: int = 40,
) -> dict[str, object]:
    """Generate one route and return tokens, frames, request metadata, validity."""
    unk_id = stoi["<UNK>"]
    eos_id = stoi["<EOS>"]
    forbidden_ids = [
        stoi["<PAD>"],
        stoi["<UNK>"],
        stoi["<BOS>"],
        stoi["<CLS>"],
        stoi["<MASK>"],
    ]

    prompt = prompt_tokens(board_prefix, angle, grouped_v)
    prompt_ids = [stoi.get(token, unk_id) for token in prompt]
    token_ids = sample_ids(
        model=model,
        prompt_ids=prompt_ids,
        device=device,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        eos_id=eos_id,
        forbidden_ids=forbidden_ids,
    )
    tokens = [itos.get(int(idx), "<UNK>") for idx in token_ids]
    validity = validity_summary(tokens, requested_board_prefix=board_prefix)

    return {
        "requested_board_prefix": board_prefix,
        "requested_angle": int(angle),
        "requested_grouped_v": int(grouped_v),
        "temperature": float(temperature),
        "top_k": None if top_k is None else int(top_k),
        "tokens": tokens,
        "sequence": " ".join(tokens),
        "frames": generated_tokens_to_frames(tokens, role_name_to_id, board_prefix=board_prefix),
        **validity,
    }

## Generating Routes

### The generation process

To generate a route, we:

1. **Create a prompt**: `<BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6>`
2. **Feed it to the model**: Get a probability distribution over the vocabulary for the next token
3. **Sample a token**: Use temperature and top-k filtering to control randomness
4. **Append and repeat**: Add the sampled token to the sequence and repeat until `<EOS>` or max length

### Temperature and top-k sampling

- **Temperature** (default 0.9): Controls randomness. Lower = more deterministic, higher = more random
- **Top-k** (default 50): Only consider the k most likely tokens. This prevents the model from generating very unlikely tokens.

These are the same techniques used in language models like GPT-3 to control output diversity.



In [ ]:
# Generate sample routes for both boards
configs = load_board_configs(["tb2", "kilter"])
configs_by_key = {config.board_key: config for config in configs}

samples = []
for board_key, config in configs_by_key.items():
    for grouped_v in [3, 5, 7]:  # V3, V5, V7
        sample = generate_one(
            model=model,
            stoi=stoi,
            itos=itos,
            device=device,
            board_prefix=config.token_prefix,
            angle=40,
            grouped_v=grouped_v,
            role_name_to_id=config.role_definitions,
            temperature=0.9,
            top_k=50,
            max_new_tokens=40,
        )
        samples.append({"board_key": board_key, **sample})

samples_df = pd.DataFrame(samples)
print("Generated route samples:")
print(samples_df[["board_key", "requested_grouped_v", "basic_valid", "sequence", "frames"]])

## Generate More Routes for Evaluation

Notebook 04 needs a larger set of generated routes for meaningful evaluation. Let's generate routes across multiple angles and grades for both boards.



In [ ]:
# Generate routes across multiple angles and grades for evaluation
all_samples = []

for board_key, config in configs_by_key.items():
    # Get common angles and grades for this board
    board_df = df_routes[df_routes["board_key"] == board_key]
    common_angles = sorted(board_df["angle"].astype(int).value_counts().head(5).index.tolist())
    common_grades = sorted(board_df["grouped_v"].astype(int).value_counts().head(8).index.tolist())
    
    print(f"\nGenerating for {config.display_name}:")
    print(f"  Angles: {common_angles}")
    print(f"  Grades: V{min(common_grades)}-V{max(common_grades)}")
    
    for angle in common_angles:
        for grade in common_grades:
            for i in range(5):  # 5 samples per condition
                sample = generate_one(
                    model=model,
                    stoi=stoi,
                    itos=itos,
                    device=device,
                    board_prefix=config.token_prefix,
                    angle=int(angle),
                    grouped_v=int(grade),
                    role_name_to_id=config.role_definitions,
                    temperature=0.9,
                    top_k=50,
                    max_new_tokens=40,
                )
                all_samples.append({"board_key": board_key, **sample})

all_samples_df = pd.DataFrame(all_samples)
print(f"\nTotal generated routes: {len(all_samples_df):,}")
print("\nBasic validity by board:")
print(all_samples_df.groupby("board_key")["basic_valid"].mean())

## Save Model and Generated Routes

We save the trained model checkpoint and generated routes for use in notebook 04 (evaluation).



In [ ]:
import os

# Save model checkpoint
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": len(stoi),
        "block_size": block_size,
        "n_embd": 128,
        "n_head": 4,
        "n_layer": 4,
        "dropout": 0.10,
        "pad_id": pad_id,
    },
    "stoi": stoi,
    "itos": {str(k): v for k, v in itos.items()},
    "best_val_loss": best_val_loss,
}
model_path = MODEL_DIR / "joint_route_gpt_generator.pth"
torch.save(checkpoint, model_path)
print(f"Saved model checkpoint to: {model_path}")

# Save training history
GEN_DIR = ROOT / "data" / "processed" / "generation"
GEN_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame(history).to_csv(GEN_DIR / "training_history.csv", index=False)
print(f"Saved training history to: {GEN_DIR / 'training_history.csv'}")

# Save generated routes (this is what notebook 04 needs)
all_samples_df.to_csv(GEN_DIR / "generated_routes.csv", index=False)
print(f"Saved {len(all_samples_df)} generated routes to: {GEN_DIR / 'generated_routes.csv'}")